In [1]:
!pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [2]:
import sys
!{sys.executable} -m pip install imbalanced-learn

You should consider upgrading via the 'c:\Users\Admin\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

import pickle
from collections import Counter
from imblearn.over_sampling import SMOTE

In [4]:
# 1. Đọc dữ liệu
df = pd.read_csv("../data/processed/iec104_processed.csv")
print(f"Kích thước ban đầu: {df.shape}")
print(f"\nThông tin dataset:")
print(df.info())

Kích thước ban đầu: (319971, 84)

Thông tin dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319971 entries, 0 to 319970
Data columns (total 84 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Flow ID                     319971 non-null  object 
 1   Src IP                      319971 non-null  object 
 2   Src Port                    319971 non-null  int64  
 3   Dst IP                      319971 non-null  object 
 4   Dst Port                    319971 non-null  int64  
 5   Protocol                    319971 non-null  int64  
 6   Timestamp                   319971 non-null  object 
 7   Flow Duration               319971 non-null  int64  
 8   Total Fwd Packet            319971 non-null  int64  
 9   Total Bwd packets           319971 non-null  int64  
 10  Total Length of Fwd Packet  319971 non-null  float64
 11  Total Length of Bwd Packet  319971 non-null  float64
 12  Fwd Packet Length M

In [ ]:
# 2. Xóa các cột không cần thiết
cols_to_drop = ["Flow ID", "Src IP", "Dst IP", "Protocol", "Timestamp"]
df = df.drop(columns=cols_to_drop, errors="ignore")
print(f"\nKích thước sau khi xóa cột: {df.shape}")


Kích thước sau khi xóa cột: (319971, 79)


In [ ]:
# 3. Kiểm tra và xử lý giá trị thiếu
missing_values = df.isnull().sum()
if missing_values.sum() > 0:
    print("Các cột có giá trị thiếu:")
    print(missing_values[missing_values > 0])
    # Xử lý giá trị thiếu (có thể dùng median hoặc mean)
    df = df.fillna(df.median(numeric_only=True))
else:
    print("Không có giá trị thiếu")

Không có giá trị thiếu


In [ ]:
# 4. Kiểm tra và xử lý giá trị vô cực
inf_cols = []
for col in df.select_dtypes(include=[np.number]).columns:
    if np.isinf(df[col]).any():
        inf_cols.append(col)
        # Thay thế inf bằng giá trị lớn nhất không phải inf
        max_val = df[col][~np.isinf(df[col])].max()
        df[col] = df[col].replace([np.inf, -np.inf], max_val)

if inf_cols:
    print(f"Đã xử lý giá trị vô cực ở các cột: {inf_cols}")
else:
    print("Không có giá trị vô cực")


Không có giá trị vô cực


In [ ]:
# 5. Mã hóa nhãn
le = LabelEncoder()
df["Label"] = le.fit_transform(df["Label"])

print("Mapping nhãn:")
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
for cls, idx in label_mapping.items():
    count = (df["Label"] == idx).sum()
    print(f"{cls} → {idx} (số lượng: {count})")

Mapping nhãn:
attackfree → 0 (số lượng: 255)
dosattack → 1 (số lượng: 304627)
floodattack → 2 (số lượng: 108)
fuzzyattack → 3 (số lượng: 939)
iec104starvationattack → 4 (số lượng: 2028)
mitmattack → 5 (số lượng: 26)
ntpddosattack → 6 (số lượng: 2278)
portscanattack → 7 (số lượng: 9710)


In [ ]:
# 6. Phân phối các lớp
print(df["Label"].value_counts().sort_index())

Label
0       255
1    304627
2       108
3       939
4      2028
5        26
6      2278
7      9710
Name: count, dtype: int64


In [ ]:
# 7. Xử lý các giá trị ngoại lai (outliers) - sử dụng IQR
print(f"\n=== Xử lý outliers ===")
X = df.drop("Label", axis=1)
y = df["Label"]

# Tính IQR cho mỗi cột
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1

# Giới hạn outliers trong khoảng [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
for col in X.columns:
    lower_bound = Q1[col] - 1.5 * IQR[col]
    upper_bound = Q3[col] + 1.5 * IQR[col]
    X[col] = X[col].clip(lower_bound, upper_bound)

print("Đã xử lý outliers bằng phương pháp IQR")


=== Xử lý outliers ===
Đã xử lý outliers bằng phương pháp IQR


In [ ]:
# 8. Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Reset index để X_scaled và y khớp nhau
X_scaled = X_scaled.reset_index(drop=True)
y = y.reset_index(drop=True)

# Kiểm tra
print(f"X_scaled: {X_scaled.shape}, y: {y.shape}")
print(f"Khớp: {X_scaled.shape[0] == len(y)}")  # Phải True

X_scaled: (319971, 78), y: (319971,)
Khớp: True


In [ ]:
# 9. Chia tập train/validation/test
print(f"\n=== Chia tập dữ liệu ===")
# Chia 70% train, 15% validation, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp  # 0.176 * 0.85 ≈ 0.15
)

print(f"Train set: {X_train.shape}, {y_train.shape}")
print(f"Validation set: {X_val.shape}, {y_val.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")



=== Chia tập dữ liệu ===
Train set: (224107, 78), (224107,)
Validation set: (47868, 78), (47868,)
Test set: (47996, 78), (47996,)


In [ ]:
# ------------------------------------------------------------------------------
# PHẦN A — SMOTE (chỉ áp dụng cho tập TRAIN, giữ 8 lớp gốc)
# ------------------------------------------------------------------------------
 
def apply_smote_safe(X_train, y_train, k_neighbors_default=5, min_samples_required=2,
                      target_min=500):
    """
    SMOTE an toàn cho các lớp cực ít mẫu (như mitmattack chỉ có ~13-18 mẫu train).
    SMOTE yêu cầu k_neighbors < số mẫu của lớp thiểu số nhỏ nhất, nên phải
    tự động hạ k_neighbors cho từng trường hợp, tránh lỗi runtime.
 
    Chỉ oversample các lớp có ít hơn `target_min` mẫu lên đúng `target_min`,
    KHÔNG ép bằng lớp đa số (dosattack) vì sẽ tạo quá nhiều mẫu tổng hợp
    không thực tế.
    """
    class_counts = Counter(y_train)
    print("Phân bố lớp TRƯỚC SMOTE:")
    for cls, cnt in sorted(class_counts.items()):
        print(f"  Lớp {cls}: {cnt} mẫu")
 
    min_class_size = min(class_counts.values())
    k_neighbors = min(k_neighbors_default, max(1, min_class_size - 1))
    if min_class_size <= min_samples_required:
        print(f"\n⚠ CẢNH BÁO: lớp nhỏ nhất chỉ có {min_class_size} mẫu train.")
        print("  SMOTE nội suy giữa các điểm lân cận nên với quá ít mẫu, các mẫu")
        print("  tổng hợp có thể không đại diện tốt cho phân phối thật.")
        print(f"  → Dùng k_neighbors={k_neighbors} (đã tự động hạ xuống).")
 
    sampling_strategy = {
        cls: target_min for cls, cnt in class_counts.items()
        if cnt < target_min
    }
 
    smote = SMOTE(
        sampling_strategy=sampling_strategy if sampling_strategy else 'auto',
        k_neighbors=k_neighbors,
        random_state=42
    )
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
 
    print("\nPhân bố lớp SAU SMOTE:")
    for cls, cnt in sorted(Counter(y_train_smote).items()):
        print(f"  Lớp {cls}: {cnt} mẫu")
 
    return X_train_smote, y_train_smote
 
 
# ------------------------------------------------------------------------------
# PHẦN B — GỘP NHÓM (dựa trên phân loại cơ chế tấn công, mục 2.1.3 luận văn)
# ------------------------------------------------------------------------------
 
GROUP_DEFINITION = {
    'attackfree': 'Normal',
    'dosattack': 'Volumetric',
    'floodattack': 'Volumetric',
    'ntpddosattack': 'Volumetric',
    'fuzzyattack': 'Protocol',
    'iec104starvationattack': 'Protocol',
    'portscanattack': 'Recon_Interception',
    'mitmattack': 'Recon_Interception',
}
 
 
def build_grouped_labels(y_encoded, label_encoder, group_definition=GROUP_DEFINITION):
    """
    Chuyển nhãn 8 lớp gốc (số nguyên 0-7) → 4 nhóm cơ chế tấn công.
    Trả về y_grouped (số nguyên 0-3) và group_encoder (LabelEncoder mới).
    """
    original_names = label_encoder.classes_
    idx_to_group = {
        idx: group_definition[name] for idx, name in enumerate(original_names)
    }
    group_str = np.array([idx_to_group[val] for val in y_encoded])
 
    group_encoder = LabelEncoder()
    y_grouped = group_encoder.fit_transform(group_str)
 
    return y_grouped, group_encoder
 
 
def print_group_distribution(y_grouped, group_encoder, set_name=""):
    print(f"\nPhân bố nhóm — {set_name}:")
    for g_idx, g_name in enumerate(group_encoder.classes_):
        cnt = np.sum(y_grouped == g_idx)
        pct = cnt / len(y_grouped) * 100
        print(f"  {g_name:<20} {cnt:>8} mẫu ({pct:>5.2f}%)")
 
 
# ==============================================================================
# CHẠY THỰC TẾ 
# ==============================================================================
 
# --- A. SMOTE trên tập train ---
print("=" * 70)
print("BƯỚC A: SMOTE TRÊN TẬP TRAIN")
print("=" * 70)
 
X_train_smote, y_train_smote = apply_smote_safe(
    X_train.values, y_train.values, target_min=500
)
 
X_train_smote_df = pd.DataFrame(X_train_smote, columns=X_train.columns)
y_train_smote_df = pd.DataFrame(y_train_smote, columns=["Label"])
 
X_train_smote_df.to_csv(output_dir + "X_train_smote.csv", index=False)
y_train_smote_df.to_csv(output_dir + "y_train_smote.csv", index=False)
 
print(f"\n✓ Đã lưu:")
print(f"  {output_dir}X_train_smote.csv  — shape: {X_train_smote_df.shape}")
print(f"  {output_dir}y_train_smote.csv  — shape: {y_train_smote_df.shape}")
 
 
# --- B. Gộp nhóm cho train / val / test (dùng label_encoder gốc `le`) ---
print("\n" + "=" * 70)
print("BƯỚC B: GỘP NHÓM (8 LỚP → 4 NHÓM)")
print("=" * 70)
 
print("\nMapping nhãn gốc → nhóm:")
for idx, name in enumerate(le.classes_):
    print(f"  {name:<25} → {GROUP_DEFINITION[name]}")
 
y_train_grouped, group_encoder = build_grouped_labels(y_train.values, le)
y_val_grouped, _ = build_grouped_labels(y_val.values, le)
y_test_grouped, _ = build_grouped_labels(y_test.values, le)
 
print(f"\n4 nhóm mới: {list(group_encoder.classes_)}")
print_group_distribution(y_train_grouped, group_encoder, "TRAIN")
print_group_distribution(y_val_grouped, group_encoder, "VALIDATION")
print_group_distribution(y_test_grouped, group_encoder, "TEST")
 
# Lưu nhãn đã gộp — X_train/X_val/X_test giữ NGUYÊN (không đổi feature,
# chỉ đổi nhãn), nên chỉ cần lưu y_*_grouped, không cần lưu lại X_*
pd.DataFrame(y_train_grouped, columns=["Label"]).to_csv(
    output_dir + "y_train_grouped.csv", index=False
)
pd.DataFrame(y_val_grouped, columns=["Label"]).to_csv(
    output_dir + "y_val_grouped.csv", index=False
)
pd.DataFrame(y_test_grouped, columns=["Label"]).to_csv(
    output_dir + "y_test_grouped.csv", index=False
)
 
with open(output_dir + "group_label_encoder.pkl", "wb") as f:
    pickle.dump(group_encoder, f)
 
print(f"\n✓ Đã lưu:")
print(f"  {output_dir}y_train_grouped.csv")
print(f"  {output_dir}y_val_grouped.csv")
print(f"  {output_dir}y_test_grouped.csv")
print(f"  {output_dir}group_label_encoder.pkl")
 
print("\n" + "=" * 70)
print("HOÀN TẤT. Tổng kết các file có sẵn để dùng cho TABNET.ipynb:")
print("=" * 70)
print("""
Kịch bản 1 — Gốc (8 lớp, không xử lý mất cân bằng):
    X_train.csv / y_train.csv          (đã có sẵn từ cell 10)
    X_val.csv / y_val.csv / X_test.csv / y_test.csv
 
Kịch bản 2 — SMOTE (8 lớp):
    X_train_smote.csv / y_train_smote.csv     (train)
    X_val.csv / y_val.csv / X_test.csv / y_test.csv   (giữ nguyên, KHÔNG đổi)
 
Kịch bản 3 — Gộp nhóm (4 lớp, không SMOTE):
    X_train.csv / y_train_grouped.csv         (train)
    X_val.csv / y_val_grouped.csv             (val)
    X_test.csv / y_test_grouped.csv           (test)
    group_label_encoder.pkl                   (dùng để đọc tên nhóm)
""")

BƯỚC A: SMOTE TRÊN TẬP TRAIN
Phân bố lớp TRƯỚC SMOTE:
  Lớp 0: 179 mẫu
  Lớp 1: 213360 mẫu
  Lớp 2: 76 mẫu
  Lớp 3: 658 mẫu
  Lớp 4: 1421 mẫu
  Lớp 5: 18 mẫu
  Lớp 6: 1595 mẫu
  Lớp 7: 6800 mẫu

Phân bố lớp SAU SMOTE:
  Lớp 0: 500 mẫu
  Lớp 1: 213360 mẫu
  Lớp 2: 500 mẫu
  Lớp 3: 658 mẫu
  Lớp 4: 1421 mẫu
  Lớp 5: 500 mẫu
  Lớp 6: 1595 mẫu
  Lớp 7: 6800 mẫu

✓ Đã lưu:
  ../data/processed/X_train_smote.csv  — shape: (225334, 78)
  ../data/processed/y_train_smote.csv  — shape: (225334, 1)

BƯỚC B: GỘP NHÓM (8 LỚP → 4 NHÓM)

Mapping nhãn gốc → nhóm:
  attackfree                → Normal
  dosattack                 → Volumetric
  floodattack               → Volumetric
  fuzzyattack               → Protocol
  iec104starvationattack    → Protocol
  mitmattack                → Recon_Interception
  ntpddosattack             → Volumetric
  portscanattack            → Recon_Interception

4 nhóm mới: ['Normal', 'Protocol', 'Recon_Interception', 'Volumetric']

Phân bố nhóm — TRAIN:
  Normal        

In [ ]:
# 10. Lưu dữ liệu đã xử lý
output_dir = "../data/processed/"

# Lưu dữ liệu
X_train.to_csv(output_dir + "X_train.csv", index=False)
X_val.to_csv(output_dir + "X_val.csv", index=False)
X_test.to_csv(output_dir + "X_test.csv", index=False)

y_train.to_csv(output_dir + "y_train.csv", index=False)
y_val.to_csv(output_dir + "y_val.csv", index=False)
y_test.to_csv(output_dir + "y_test.csv", index=False)

# Lưu scaler và label encoder
import pickle
with open(output_dir + "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
    
with open(output_dir + "label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Đã lưu tất cả files:")
print("- X_train.csv, X_val.csv, X_test.csv")
print("- y_train.csv, y_val.csv, y_test.csv")
print("- scaler.pkl, label_encoder.pkl")


Đã lưu tất cả files:
- X_train.csv, X_val.csv, X_test.csv
- y_train.csv, y_val.csv, y_test.csv
- scaler.pkl, label_encoder.pkl


In [ ]:
# 11. Thống kê cuối cùng
print(f"Số features: {X_train.shape[1]}")
print(f"Số classes: {len(np.unique(y))}")
print(f"\nPhân phối nhãn trong train set:")
print(y_train.value_counts().sort_index())
print(f"\nPhân phối nhãn trong validation set:")
print(y_val.value_counts().sort_index())
print(f"\nPhân phối nhãn trong test set:")
print(y_test.value_counts().sort_index())

Số features: 78
Số classes: 8

Phân phối nhãn trong train set:
Label
0       179
1    213360
2        76
3       658
4      1421
5        18
6      1595
7      6800
Name: count, dtype: int64

Phân phối nhãn trong validation set:
Label
0       38
1    45573
2       16
3      140
4      303
5        4
6      341
7     1453
Name: count, dtype: int64

Phân phối nhãn trong test set:
Label
0       38
1    45694
2       16
3      141
4      304
5        4
6      342
7     1457
Name: count, dtype: int64


In [ ]:
import pickle
import pandas as pd

with open("../data/processed/scaler.pkl", "rb") as f:
    sc = pickle.load(f)

X_test = pd.read_csv("../data/processed/X_test.csv")

# Lấy 1 dòng bất kỳ, scale rồi predict thử
from pytorch_tabnet.tab_model import TabNetClassifier
import numpy as np

model = TabNetClassifier()
model.load_model("../data/processed/tabnet_iec104_model.zip.zip")

row = X_test.iloc[[0]]
X_scaled = sc.transform(row)
pred = model.predict(X_scaled)
proba = model.predict_proba(X_scaled)

with open("../data/processed/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

print("Predicted class index:", pred[0])
print("Predicted label:", le.classes_[pred[0]])
print("Confidence:", round(float(proba[0][pred[0]]) * 100, 2), "%")
print("All probas:", {le.classes_[i]: round(float(proba[0][i])*100,2) for i in range(len(le.classes_))})

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


Predicted class index: 3
Predicted label: fuzzyattack
Confidence: 53.31 %
All probas: {'attackfree': 44.68, 'dosattack': 0.21, 'floodattack': 0.43, 'fuzzyattack': 53.31, 'iec104starvationattack': 0.43, 'mitmattack': 0.07, 'ntpddosattack': 0.21, 'portscanattack': 0.66}
